# 01 — Data Prep

Master Excel'i (`data/master/raw2value_v4.xlsx`) `extract_master_excel` ile parquet referans tablolarına dönüştürür ve her çıktıyı doğrular.

In [ ]:
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    # Jupyter kernelinin OutStream'inde reconfigure yok; sorun yok
    pass

In [ ]:
import random
random.seed(42)
import numpy as np
np.random.seed(42)

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import pickle

# notebook ml/notebooks/ içinden çalıştığı için repo köküne çık
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('cwd:', os.getcwd())

In [ ]:
from ml.src.etl import extract_master_excel

result = extract_master_excel('data/master/raw2value_v4.xlsx', 'data/reference/')
for name, df in result.items():
    print(f'{name:25s} shape={df.shape}')

## Her parquet'i geri okuyup doğrula

In [ ]:
outputs = [
    'materials.parquet',
    'routes.parquet',
    'organizations.parquet',
    'buyer_markets.parquet',
    'distances.parquet',
    'carbon.parquet',
    'carbon_alternatives.parquet',
    'fx_rates.parquet',
    'risk_delivery.parquet',
    'risk_price.parquet',
    'quality_match.parquet',
    'quality_demand.parquet',
]
for fn in outputs:
    print('=' * 70)
    print(fn)
    print('=' * 70)
    df = pd.read_parquet(f'data/reference/{fn}')
    print('shape:', df.shape)
    print('dtypes:')
    print(df.dtypes)
    print('head(2):')
    print(df.head(2))
    print()

In [ ]:
with open('data/reference/distance_lookup.pkl', 'rb') as f:
    distance_lookup = pickle.load(f)
print('distance_lookup entries:', len(distance_lookup))
sample_key = next(iter(distance_lookup))
print('sample key  :', sample_key)
print('sample value:', distance_lookup[sample_key])

## Kabul kriterleri

In [ ]:
routes = pd.read_parquet('data/reference/routes.parquet')
assert routes.shape[0] == 15, f'routes 15 satir bekleniyor, {routes.shape[0]} bulundu'

distances = pd.read_parquet('data/reference/distances.parquet')
assert distances['Distance km'].dtype.kind == 'f', f'Distance km float olmali, {distances["Distance km"].dtype}'

carbon = pd.read_parquet('data/reference/carbon.parquet')
assert carbon.shape[0] == 4, f'carbon 4 satir bekleniyor, {carbon.shape[0]}'

qm = pd.read_parquet('data/reference/quality_match.parquet')
assert qm.shape[0] == 9

print('Tum kabul kriterleri OK')